# 从零实现 Node2Vec：二阶随机游走、Skip-Gram 与时间评估

本 Notebook 用纯 Python/PyTorch 手写 Node2Vec 的二阶 biased walk、skip-gram negative sampling、embedding 训练、链接预测与节点线性评估；不调用 gensim、NetworkX Node2Vec、PyG、DGL 或现成图嵌入实现。

参考：[node2vec, KDD 2016](https://arxiv.org/abs/1607.00653)、[word2vec negative sampling](https://arxiv.org/abs/1310.4546)。合成图结果只用于回归实现与协议，不等价于真实网络上的泛化性能。


In [ ]:
from __future__ import annotations
import copy, hashlib, json, math, random, warnings
from dataclasses import dataclass
from types import MappingProxyType
import numpy as np
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)
import torch
from torch import nn
import torch.nn.functional as F

SEED = 6201
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)

def digest(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",",":"), ensure_ascii=False).encode()).hexdigest()

def tensor_desc(state):
    out={}
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous()
        out[k]={"dtype":str(v.dtype),"shape":list(v.shape),"sha256":hashlib.sha256(v.numpy().tobytes()).hexdigest()}
    return out

assert torch.get_num_threads() == 1
assert digest({"x":1}) != digest({"x":2})
assert not any(k in globals() for k in ("gensim","networkx","torch_geometric","dgl"))


## 1. 图合同：有向、无向与时间不能含糊

`TemporalGraph` 的边记录为 `(src,dst,time)`。无向图在邻接表中自动展开两个方向，但语义摘要仍保留规范化后的无向边；有向图只允许沿 `src→dst` 走。重复边、越界节点、非有限时间和自环都 fail closed。

时间切分以 `time <= cutoff` 为训练快照。训练游走绝不能看到未来边；评估负例则从“全时间真边”中过滤，避免把未来会出现的正边当作 false negative。生产中若未来真值未知，应明确报告候选负例可能含潜在正边。


In [ ]:
@dataclass(frozen=True)
class TemporalGraph:
    num_nodes: int
    edges: tuple[tuple[int,int,float], ...]
    directed: bool = False

    def validate(self):
        if not isinstance(self.num_nodes,int) or self.num_nodes < 1 or not self.edges:
            raise ValueError("图必须有节点和边")
        seen=set()
        for u,v,t in self.edges:
            if not (isinstance(u,int) and isinstance(v,int) and 0 <= u < self.num_nodes and 0 <= v < self.num_nodes):
                raise ValueError("节点索引非法")
            if u == v or not math.isfinite(float(t)):
                raise ValueError("拒绝自环或非有限时间")
            key=(u,v,t) if self.directed else (min(u,v),max(u,v),t)
            if key in seen: raise ValueError("重复边")
            seen.add(key)
        return self

    def snapshot(self, cutoff: float):
        self.validate()
        kept=tuple(e for e in self.edges if e[2] <= cutoff)
        if not kept: raise ValueError("快照为空")
        return TemporalGraph(self.num_nodes, kept, self.directed)

    def adjacency(self):
        self.validate(); adj=[[] for _ in range(self.num_nodes)]
        for u,v,_ in self.edges:
            adj[u].append(v)
            if not self.directed: adj[v].append(u)
        return tuple(tuple(sorted(set(row))) for row in adj)

    def semantic(self):
        normalized=sorted((u,v,t) if self.directed else (min(u,v),max(u,v),t) for u,v,t in self.edges)
        return {"num_nodes":self.num_nodes,"directed":self.directed,"edges":normalized}

ug=TemporalGraph(4,((0,1,1.),(1,2,2.),(2,3,5.)),False).validate()
dg=TemporalGraph(4,((0,1,1.),(1,2,2.),(2,3,5.)),True).validate()
assert ug.adjacency()[1] == (0,2) and dg.adjacency()[1] == (2,)
assert ug.snapshot(2).adjacency()[2] == (1,)
assert digest(ug.semantic()) != digest(dg.semantic())


## 2. Node2Vec 的二阶转移概率

已从 $t$ 走到 $v$，候选下一节点为 $x$。对无向图，本实现使用：

$$\pi_{vx}=w_{vx}\alpha_{pq}(t,x),\quad
\alpha_{pq}(t,x)=\begin{cases}1/p & x=t\\1 & x\in N(t)\\1/q & \text{otherwise}\end{cases}.$$

所有边权为 1。$p<1$ 鼓励立即返回，$q<1$ 鼓励向外探索；`p,q` 必须有限且大于 0。对有向图，“距离 1”明确解释为存在 `t→x`，而不是悄悄对称化。第一次跳转没有前驱，按当前节点邻居均匀采样。


In [ ]:
def transition_probs(adj, previous: int|None, current: int, p: float, q: float):
    if not (math.isfinite(p) and math.isfinite(q) and p > 0 and q > 0):
        raise ValueError("p/q 必须有限且为正")
    candidates=adj[current]
    if not candidates: return (), torch.empty(0)
    if previous is None:
        weights=torch.ones(len(candidates),dtype=torch.float64)
    else:
        previous_neighbors=set(adj[previous]); vals=[]
        for x in candidates:
            vals.append(1/p if x == previous else (1.0 if x in previous_neighbors else 1/q))
        weights=torch.tensor(vals,dtype=torch.float64)
    return candidates, weights/weights.sum()

triangle_tail=TemporalGraph(4,((0,1,1.),(1,2,1.),(0,2,1.),(1,3,1.)),False)
adj=triangle_tail.adjacency()
cand,prob=transition_probs(adj,0,1,p=2.,q=.5)
# 当前 1 的候选为 0/2/3：返回 0.5、邻居 1、远点 2，总和 3.5
expected=torch.tensor([.5/3.5,1/3.5,2/3.5],dtype=torch.float64)
assert cand == (0,2,3) and torch.allclose(prob,expected)
_, return_heavy=transition_probs(adj,0,1,p=.01,q=10.)
_, outward_heavy=transition_probs(adj,0,1,p=10.,q=.01)
assert return_heavy[0] > .98 and outward_heavy[-1] > .98
try:
    transition_probs(adj,0,1,0,1)
    raise AssertionError("p=0 未拒绝")
except ValueError as exc: assert "p/q" in str(exc)


## 3. 可复现 walk 与孤立点语义

游走每一步都从显式 generator 的 `torch.multinomial` 采样。同 seed、同图、同起点必须逐 token 一致。孤立节点的 walk 只包含起点并立即结束；不能为了凑长度虚构自环，因为那会改变训练分布。

单条 walk 的时间复杂度为 $O(L\bar d)$（此处每步现算候选权重）；生产实现可为每条有向边预计算 alias table，把采样摊还到 $O(1)$，代价是 $O(E)$ 到 $O(E\bar d)$ 的额外索引内存。


In [ ]:
def node2vec_walk(graph: TemporalGraph, start: int, length: int, p: float, q: float,
                  generator: torch.Generator) -> list[int]:
    graph.validate()
    if not 0 <= start < graph.num_nodes or length < 1: raise ValueError("start/length 非法")
    adj=graph.adjacency(); walk=[start]; previous=None; current=start
    while len(walk) < length:
        candidates,probs=transition_probs(adj,previous,current,p,q)
        if not candidates: break
        index=int(torch.multinomial(probs.float(),1,generator=generator))
        nxt=candidates[index]; walk.append(nxt); previous,current=current,nxt
    return walk

w1=node2vec_walk(triangle_tail,0,8,1.,1.,torch.Generator().manual_seed(9))
w2=node2vec_walk(triangle_tail,0,8,1.,1.,torch.Generator().manual_seed(9))
assert w1 == w2 and len(w1) == 8
isolated_graph=TemporalGraph(5,triangle_tail.edges,False)
assert node2vec_walk(isolated_graph,4,10,1,1,torch.Generator().manual_seed(1)) == [4]
assert node2vec_walk(dg,3,5,1,1,torch.Generator().manual_seed(1)) == [3]


## 4. `degree^0.75`、无放回且时间一致的负采样

窗口半径为 $w$ 时，每条 walk 产生最多 $O(Lw)$ 个有序 `(center,context)`。本实现令节点负采样权重为训练快照度数的 $d(v)^{3/4}$，对每个正 pair 先过滤 self、正 context 和当前可见真邻居，再按权重无放回抽取 $K$ 个负例；候选或正权重不足时 fail closed，而不是重复同一节点凑数。

时间语义必须分两条路径：训练 sampler 只能读取 `time<=cutoff` 的邻接和度数，未来边可能成为不可避免的潜在假负例；离线评估在标签已经揭晓后，才可用全时间真边过滤候选，避免把 holdout 正边计作负例。把 full graph 提前传给训练 sampler 虽然“更干净”，实质上泄漏了未来标签。


In [ ]:
def walks_to_pairs(walks: list[list[int]], window: int) -> torch.Tensor:
    if window < 1: raise ValueError("window 必须为正")
    pairs=[]
    for walk in walks:
        for i,c in enumerate(walk):
            for j in range(max(0,i-window),min(len(walk),i+window+1)):
                if i != j: pairs.append((c,walk[j]))
    if not pairs: raise ValueError("没有可训练 pair")
    return torch.tensor(pairs,dtype=torch.long)

def degree_weights62(graph: TemporalGraph) -> torch.Tensor:
    degree = torch.tensor([len(row) for row in graph.adjacency()], dtype=torch.float64)
    return degree.pow(.75)

def sample_negatives(centers: torch.Tensor, positive: torch.Tensor, num_nodes: int,
                     forbidden_neighbors: tuple[frozenset,...], node_weights: torch.Tensor,
                     k: int, generator: torch.Generator) -> torch.Tensor:
    if centers.shape != positive.shape or centers.dtype != torch.long or k < 1:
        raise ValueError("负采样合同非法")
    if node_weights.shape != (num_nodes,) or not torch.isfinite(node_weights).all() or bool((node_weights < 0).any()):
        raise ValueError("负采样权重非法")
    rows=[]
    for c,pos in zip(centers.tolist(),positive.tolist()):
        allowed=[n for n in range(num_nodes)
                 if n != c and n != pos and n not in forbidden_neighbors[c] and float(node_weights[n]) > 0]
        if len(allowed) < k:
            raise ValueError("过滤后正权重负样本不足，禁止有放回补齐")
        weights=node_weights[allowed]
        chosen=torch.multinomial(weights,k,replacement=False,generator=generator)
        rows.append(torch.tensor([allowed[i] for i in chosen.tolist()],dtype=torch.long))
    return torch.stack(rows)

pairs_probe=walks_to_pairs([[0,1,2]],1)
assert pairs_probe.tolist() == [[0,1],[1,0],[1,2],[2,1]]

# 边 (0,3,t=5) 是未来真边，节点 3 在训练快照另有边，因此具备正的 degree^.75 权重。
future_full62=TemporalGraph(5,((0,1,1.),(3,4,1.),(0,3,5.)),False).validate()
future_train62=future_full62.snapshot(1.)
train_truth_probe62=tuple(frozenset(row) for row in future_train62.adjacency())
eval_truth_probe62=tuple(frozenset(row) for row in future_full62.adjacency())
train_sample_probe62=sample_negatives(torch.tensor([0]),torch.tensor([1]),5,train_truth_probe62,
                                      degree_weights62(future_train62),2,torch.Generator().manual_seed(3))
eval_sample_probe62=sample_negatives(torch.tensor([0]),torch.tensor([1]),5,eval_truth_probe62,
                                     degree_weights62(future_full62),1,torch.Generator().manual_seed(3))
assert 3 in train_sample_probe62[0].tolist()  # 训练不知道未来，不能提前过滤
assert 3 not in eval_sample_probe62[0].tolist()  # 评估已知全时间真值，必须过滤
assert train_sample_probe62.unique().numel()==2
try:
    sample_negatives(torch.tensor([1]),torch.tensor([0]),3,
                     (frozenset({1}),frozenset({0,2}),frozenset({1})),torch.ones(3),1,
                     torch.Generator().manual_seed(1))
    raise AssertionError("无负例候选未报错")
except ValueError as exc: assert "不足" in str(exc)


## 5. 手写 embedding、负采样目标与评估头

Skip-Gram negative sampling 最大化

$$\log\sigma(u_c^\top v_o)+\sum_{k=1}^{K}\log\sigma(-u_c^\top v_{n_k}).$$

输入与上下文 embedding 分开参数化。推理通常取二者平均或只取输入表，本例取输入表。`LinkScorer` 用归一化点积，`NodeLinearProbe` 只在线性评估阶段接收标签。


In [ ]:
class Node2VecSkipGram(nn.Module):
    def __init__(self,num_nodes:int,dim:int):
        super().__init__(); self.input=nn.Embedding(num_nodes,dim); self.context=nn.Embedding(num_nodes,dim)
        nn.init.normal_(self.input.weight,std=.2); nn.init.zeros_(self.context.weight)
    def forward(self,center:torch.Tensor,positive:torch.Tensor,negative:torch.Tensor):
        if center.ndim!=1 or positive.shape!=center.shape or negative.ndim!=2 or len(negative)!=len(center):
            raise ValueError("skip-gram batch shape 非法")
        u=self.input(center); pos=(u*self.context(positive)).sum(-1)
        neg=torch.einsum("bd,bkd->bk",u,self.context(negative))
        return -(F.logsigmoid(pos)+F.logsigmoid(-neg).sum(1)).mean()
    def embeddings(self): return self.input.weight

class LinkScorer(nn.Module):
    def forward(self,z:torch.Tensor,edges:torch.Tensor):
        if edges.ndim!=2 or edges.shape[0]!=2: raise ValueError("edges 必须 [2,E]")
        zn=F.normalize(z,dim=-1); return (zn[edges[0]]*zn[edges[1]]).sum(-1)

class NodeLinearProbe(nn.Module):
    def __init__(self,dim:int,classes:int): super().__init__(); self.linear=nn.Linear(dim,classes)
    def forward(self,z:torch.Tensor): return self.linear(z)

sg_probe=Node2VecSkipGram(4,8)
assert sg_probe(torch.tensor([0]),torch.tensor([1]),torch.tensor([[2,3]])).ndim==0
assert LinkScorer()(torch.eye(3),torch.tensor([[0,0],[0,1]])).tolist()==[1.,0.]
assert NodeLinearProbe(8,2)(torch.zeros(3,8)).shape==(3,2)


## 6. 受控社区图训练

图由两个社区构成，训练快照内各自接近 clique，并留出两条较晚出现的边做时间链接评估。walk 只在 `cutoff=2` 的快照生成；负例过滤却使用全时间真边。节点分类按节点 ID 划分 train/test，因此只是小型协议测试，真实场景应按未来时间、新节点或整社区做 inductive split。


In [ ]:
edges62=[]
for base in (0,5):
    for i in range(base,base+5):
        for j in range(i+1,base+5):
            if (i,j) not in ((0,4),(5,9)): edges62.append((i,j,1.0+(i+j)%2))
edges62 += [(0,4,3.0),(5,9,3.0)]
full62=TemporalGraph(10,tuple(edges62),False).validate(); train62=full62.snapshot(2.0)
walks=[]
walk_gen=torch.Generator().manual_seed(SEED)
for _ in range(8):
    for node in range(10): walks.append(node2vec_walk(train62,node,10,.75,1.5,walk_gen))
pairs62=walks_to_pairs(walks,2)
train_truth62=tuple(frozenset(row) for row in train62.adjacency())
eval_truth62=tuple(frozenset(row) for row in full62.adjacency())
train_negative_weights62=degree_weights62(train62)
model62=Node2VecSkipGram(10,12); opt=torch.optim.Adam(model62.parameters(),lr=.03)
losses=[]
for step in range(35):
    idx=torch.randint(len(pairs62),(128,),generator=torch.Generator().manual_seed(SEED+step))
    batch=pairs62[idx]
    negatives=sample_negatives(batch[:,0],batch[:,1],10,train_truth62,train_negative_weights62,3,torch.Generator().manual_seed(9000+step))
    loss=model62(batch[:,0],batch[:,1],negatives)
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(float(loss))

z=model62.embeddings().detach(); probe62=NodeLinearProbe(12,2); probe_opt=torch.optim.Adam(probe62.parameters(),lr=.08)
labels62=torch.tensor([0]*5+[1]*5); train_nodes=torch.tensor([0,1,2,5,6,7]); test_nodes=torch.tensor([3,4,8,9])
for _ in range(40):
    loss=F.cross_entropy(probe62(z[train_nodes]),labels62[train_nodes]); probe_opt.zero_grad(); loss.backward(); probe_opt.step()
node_acc62=float((probe62(z[test_nodes]).argmax(1)==labels62[test_nodes]).float().mean())

heldout=torch.tensor([[0,5],[4,9]])
negative_edges=torch.tensor([[0,1],[9,8]])  # 跨社区非边
link=LinkScorer(); pos_score=link(z,heldout); neg_score=link(z,negative_edges)
link_pair_acc62=float(((pos_score[:,None] > neg_score[None,:]).float().mean()))
assert losses[-1] < losses[0] and math.isfinite(losses[-1])
assert node_acc62 >= .75 and link_pair_acc62 >= .75
assert not any(t>2 for _,_,t in train62.edges)
assert 4 not in train_truth62[0] and 4 in eval_truth62[0]
print({"sg_first":round(losses[0],4),"sg_last":round(losses[-1],4),
       "controlled_node_acc":node_acc62,"controlled_link_pair_acc":link_pair_acc62})


## 7. 训练/推理复杂度与典型失败

若有 $R$ 条 walk、长度 $L$、窗口 $w$、负例数 $K$、维度 $D$，pair 数约 $O(RLw)$，训练计算约 $O(RLwKD)$。全量 pair 常驻内存会爆炸，生产应流式生成、分片 shuffle、异步 alias sampler，并固定 worker seed。

典型失败：把无向边只存一侧；有向图却偷偷对称化；训练 walk 看见时间 holdout；负采样命中未来真边；孤立点虚构自环；`p/q` 与论文定义相反；同 seed 因多 worker 调度不可复现；链接评估把反向边当负例；transductive 节点结果冒充 inductive 泛化。


## 8. 发布制品与输入语义重算

manifest 绑定 directedness、完整图、训练 cutoff/split、walk 的 `p/q/length/window`、negative filtering recipe 以及 state 的 key/dtype/shape/bytes。`PublishedNode2Vec` 接收真实 `TemporalGraph` 后重新计算其语义，只为注册图返回 embedding。

包内 manifest 即使整体重签，也无法改变只读带外 registry 的 release 摘要。生产实现需把 registry 替换成签名公钥/KMS 与不可变制品仓，并提供吊销和回滚。


In [ ]:
WALK62={"p":.75,"q":1.5,"length":10,"window":2,"walks_per_node":8,"rng":"torch.Generator"}
SPLIT62={"cutoff":2.0,"train_if":"time<=cutoff","heldout":[[0,4,3.0],[5,9,3.0]]}
NEG62={"k":3,"weight":"train_snapshot_degree^0.75","replacement":False,"train_filter":"self+positive+train_snapshot_neighbors","evaluation_filter":"self+positive+all-time-neighbors"}
state62={k:v.detach().cpu().clone() for k,v in model62.state_dict().items()}
manifest62={"schema":"temporal-edge(src,dst,time)/v1","graph":full62.semantic(),"train_graph":train62.semantic(),
            "split":SPLIT62,"walk":WALK62,"negative":NEG62,"config":{"num_nodes":10,"dim":12},"state":tensor_desc(state62)}
artifact62={"release_id":"node2vec-62-v1","manifest":manifest62,"state":state62}
def artifact_digest62(a): return digest({"release_id":a["release_id"],"manifest":a["manifest"]})
_TRUSTED_RELEASES62=MappingProxyType({"node2vec-62-v1":artifact_digest62(artifact62)})

class PublishedNode2Vec(nn.Module):
    def __init__(self,model,graph_semantic): super().__init__(); self.model=model.eval(); self.graph_semantic=graph_semantic
    def forward(self,graph:TemporalGraph,nodes:torch.Tensor):
        if digest(graph.validate().semantic()) != digest(self.graph_semantic): raise ValueError("输入图语义未注册")
        if nodes.dtype!=torch.long or nodes.ndim!=1 or nodes.numel()==0 or int(nodes.min())<0 or int(nodes.max())>=graph.num_nodes:
            raise ValueError("nodes 非法")
        with torch.no_grad(): return self.model.embeddings()[nodes]

def load62(a):
    rid=a.get("release_id")
    if rid not in _TRUSTED_RELEASES62 or artifact_digest62(a)!=_TRUSTED_RELEASES62[rid]: raise ValueError("带外 registry 拒绝 release")
    if a["manifest"]["state"] != tensor_desc(a["state"]): raise ValueError("state 描述不匹配")
    cfg=a["manifest"]["config"]; m=Node2VecSkipGram(**cfg); m.load_state_dict(a["state"],strict=True)
    return PublishedNode2Vec(m,a["manifest"]["graph"])

published62=load62(artifact62)
assert published62(full62,torch.tensor([0,9])).shape==(2,12)
attack62=copy.deepcopy(artifact62); attack62["manifest"]["walk"]["p"]=99.; attack62["manifest"]["state"]=tensor_desc(attack62["state"])
try:
    load62(attack62); raise AssertionError("整体重签攻击未拒绝")
except ValueError as exc: assert "registry" in str(exc)
try:
    published62(train62,torch.tensor([0])); raise AssertionError("快照冒充注册图未拒绝")
except ValueError as exc: assert "语义" in str(exc)


## 9. 生产检查清单

需要额外覆盖：加权/多重边与方向语义；动态图增量 alias table；高阶节点和孤立节点采样公平性；多进程 RNG；时间一致的负例定义；embedding 漂移与冷启动；ANN 召回误差；多 seed 统计；隐私删除传播；制品签名、词表/节点 ID 映射版本与回滚。

本 Notebook 的结论严格限定为：精确小图转移概率、极端 `p/q`、seed、孤立点、时间泄漏与 false negative oracle 均通过，且受控社区图训练可用；它没有证明参数会迁移到未知业务图。


In [ ]:
assert set(manifest62)=={"schema","graph","train_graph","split","walk","negative","config","state"}
assert full62.semantic()["directed"] is False
assert node_acc62 >= .75 and link_pair_acc62 >= .75
print("Node2Vec 62：转移、时间切分、训练、评估与发布 oracle 通过。")
